# Meta-características

As meta-características são as **clássicas do `pymfe`**: `MFE()` com os grupos padrão (general, statistical, info-theory,
model-based, landmarking; resumo mean/sd), como nas atividades anteriores. Cada dataset é padronizado (z-score por atributo) e,
se tiver mais de 5.000 instâncias, subamostrado de forma estratificada. Esta meta-task não tem meta-características específicas do
domínio pois os 120 datasets são de classificação clássica, então só as clássicas são usadas.

A extração (`src/metafeatures.py`) é feita para cada dataset. O resultado fica em cache em
`representations/PyMFE.csv` (`FORCE_PYMFE = True` para reextrair: dá aprox. 7 min).

> Descartei colunas com 5% de NaN/inf, imputação pela mediana, remoção de colunas constantes e normalização min-max. O alvo é então projetado nessa escala e o vizinho mais próximo é o de menor distância euclidiana entre os 119.

In [1]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "src" / "config.py").exists():
    ROOT = Path(r"D:\Projetos\DisciplinaMtL\entrega3")
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as cfg, search, analysis as an
from src.metatask_data import load_meta_task

an.setup_matplotlib()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
cfg.make_dirs()

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(cfg.FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight", facecolor="white")

def save_table(df, name, index=True):
    df.to_csv(cfg.TABLES_DIR / f"{name}.csv", index=index)

from src import metafeatures as mf

FORCE_PYMFE = False
meta = load_meta_task()
pymfe = mf.load_or_extract_pymfe(meta.names, force=FORCE_PYMFE)
nan_cols = pymfe.replace([np.inf, -np.inf], np.nan).isna().any()
print(f"PyMFE: {pymfe.shape[0]} datasets x {pymfe.shape[1]} meta-características brutas; "
      f"{int(nan_cols.sum())} colunas têm NaN/inf em algum dataset")
grupos = pd.Series([c.split(".")[0] for c in pymfe.columns]).value_counts()
display(grupos.rename("nº de colunas").to_frame().T)

PyMFE: 120 datasets x 111 meta-características brutas; 23 colunas têm NaN/inf em algum dataset


,attr_conc,attr_ent,best_node,can_cor,class_conc,cor,cov,eigenvalues,elite_nn,freq_class,g_mean,h_mean,iq_range,joint_ent,kurtosis,...,nr_bin,nr_cat,nr_class,nr_cor_attr,nr_disc,nr_inst,nr_norm,nr_num,nr_outliers,ns_ratio,num_to_cat,p_trace,roy_root,sd_ratio,w_lambda
nº de colunas,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [2]:
ctx = search.build_context(meta)               # vizinhos em LODO: normalização ajustada nos 119 restantes, por alvo
print(f"Colunas usadas após a preparação LODO: {ctx['dims'].min()}–{ctx['dims'].max()} (de {pymfe.shape[1]})")
nn = pd.DataFrame({"vizinho mais próximo": [meta.names[j] for j in ctx["neighbors"][:, 0]],
                   "2º": [meta.names[j] for j in ctx["neighbors"][:, 1]],
                   "3º": [meta.names[j] for j in ctx["neighbors"][:, 2]],
                   "distância ao 1º": ctx["distances"][:, 0]}, index=meta.names)
save_table(nn, "vizinhos_pymfe")
display(nn.head(10).round(3))

Colunas usadas após a preparação LODO: 102–102 (de 111)


,vizinho mais próximo,2º,3º,distância ao 1º
abalone,waveform,wine-quality-white,magic,1.800
acute-inflammation,acute-nephritis,statlog-heart,heart-hungarian,1.379
acute-nephritis,acute-inflammation,mushroom,molec-biol-promoter,1.509
adult,bank,magic,statlog-german-credit,1.597
annealing,horse-colic,cardiotocography-3clases,lymphography,1.870
arrhythmia,musk-1,low-res-spect,flags,3.014
audiology-std,soybean,dermatology,arrhythmia,143.071
balance-scale,hayes-roth,pima,car,1.427
balloons,tic-tac-toe,molec-biol-promoter,balance-scale,1.638
bank,statlog-german-credit,statlog-australian-credit,adult,1.415
